In [2]:
## requirements ##
import chess.pgn
import random
import os
from dotenv import dotenv_values
from itertools import count
import shutil
import pandas as pd

In [3]:
## data paths ##
DATA_ROOT = os.environ['DATA_ROOT']
DATA_JAN2023 = os.path.join(
    DATA_ROOT,
    "lichess-01-23"
)
DATA_SAMPLES = os.path.join(
    DATA_JAN2023,
    "samples"
)
MAIN_DATA_FILE = os.path.join(
    DATA_JAN2023,
    "lichess_db_standard_rated_2023-01.pgn"
)
FULL_DATASET_DIR = os.path.join(
    DATA_JAN2023,
    "full"
)
TINY_SAMPLE_FILE = os.path.join(
    DATA_SAMPLES,
    "tiny_sample.parquet"
)
SMALL_SAMPLE_FILE = os.path.join(
    DATA_SAMPLES,
    "small_sample.parquet"
)
MEDIUM_SAMPLE_FILE = os.path.join(
    DATA_SAMPLES,
    "medium_sample.parquet"
)
LARGE_SAMPLE_FILE = os.path.join(
    DATA_SAMPLES,
    "large_sample.parquet"
)
HUGE_SAMPLE_FILE = os.path.join(
    DATA_SAMPLES,
    "huge_sample.parquet"
)
print("DATA ROOT PATH: ", DATA_ROOT)
print("DATA JAN2023 PATH: ", DATA_JAN2023)
print("DATA SAMPLES PATH: ", DATA_SAMPLES)


DATA ROOT PATH:  /Users/ashishneupane/data
DATA JAN2023 PATH:  /Users/ashishneupane/data/lichess-01-23
DATA SAMPLES PATH:  /Users/ashishneupane/data/lichess-01-23/samples


In [4]:
def break_pgn(input_file, output_dir, max_games=10e6):
    """
    Break a large pgn file into smaller files with one game per file
    :param input_file: Path to input file with games separated by game_delimiter
    :param output_dir: Path to output directory
    :param game_delimiter: Delimiter between games in the input file, default "\n\n"

    :return: None
    """
    if (not os.path.exists(output_dir)):
        os.makedirs(output_dir)

    with open(input_file) as file:
        for i in count():
            if i > max_games:
                break
            firstline = next(file, None)
            if firstline is None:
                break
            with open(os.path.join(output_dir, f"{i}.pgn"), 'w') as out:
                out.write(firstline)
                for line in file:
                    out.write(line)
                    if line.startswith("1"):
                        line = next(file, None)
                        break

In [5]:
def downsample_pgn(input_dir, output_file, sample_size=None, sample_ratio=None, seed=None):
    """
    Downsample a folder containing pgn files where each file is a single game.
    :param input_file: Path to input file
    :param output_file: Path to output file
    :param sample_size: Number of games to sample, if None, sample_ratio is used
    :param sample_ratio: Ratio of games to sample, sample size takes precedence over sample ratio
    :param seed: Random seed, if None, no seed is set

    :return: None
    """
    # count the number of .pgn files in the input directory
    num_files = len([f for f in os.listdir(input_dir) if f.endswith('.pgn')])
    print("Number of files in input directory: ", num_files)

    # set sample size and seed
    if (sample_size is None) and (sample_ratio is None):
        # If no sample size or ratio is specified, sample 10 games
        sample_size = 10
    elif sample_size is None:
        # If no sample size is specified, sample a ratio of the games
        sample_size = int(num_files * sample_ratio)
    if seed is not None:
        # Set random seed
        random.seed(seed)

    # Sample random games from a list of 1 to num_files
    sampled_games = random.sample(range(1, num_files+1), sample_size)

    # load the sample games and build a pandas dataframe of samples to write as a parquet file
    games_dict = []
    for sampleGameNum in sampled_games:
        games_dict.append(load_game_dict(os.path.join(input_dir, f"{sampleGameNum}.pgn")))
    df = pd.DataFrame(games_dict)
    df.to_parquet(output_file, engine="fastparquet")

def load_game_dict(pgn_filename):
    """ 
    Load a game from a pgn file and return a dictionary of the game headers and moves
    :param pgn_filename: Path to pgn file

    :return: Dictionary of game headers and moves. Each header is a key, along with "moves"
    """
    game = chess.pgn.read_game(open(pgn_filename, "r"))
    data_dict = {}
    for key in game.headers.keys():
        data_dict[key] = game.headers[key]
    data_dict["moves"] = str(game.mainline_moves())
    return data_dict

In [ ]:
## break main file into smaller files with one game per file
#break_pgn(MAIN_DATA_FILE, FULL_DATASET_DIR)

In [6]:
## tiny by sample size ##
TINY_SAMPLE_SIZE = 1000

downsample_pgn(FULL_DATASET_DIR, TINY_SAMPLE_FILE, sample_size=TINY_SAMPLE_SIZE)

Number of files in input directory:  4974110


In [7]:
# larger by sample ratio ##
SMALL_SAMPLE_RATIO = 0.01
MEDIUM_SAMPLE_RATIO = 0.05
LARGE_SAMPLE_RATIO = 0.1
HUGE_SAMPLE_RATIO = 0.2

SEED = 42

for sample_ratio, sample_dir in zip(
    [SMALL_SAMPLE_RATIO, MEDIUM_SAMPLE_RATIO, LARGE_SAMPLE_RATIO, HUGE_SAMPLE_RATIO],
    [SMALL_SAMPLE_FILE, MEDIUM_SAMPLE_FILE, LARGE_SAMPLE_FILE, HUGE_SAMPLE_FILE]
):
    downsample_pgn(FULL_DATASET_DIR, sample_dir, sample_ratio=sample_ratio, seed=SEED)

Number of files in input directory:  4974110
Number of files in input directory:  4974110
Number of files in input directory:  4974110
Number of files in input directory:  4974110
